**背景：**

提升奖励模型在通用领域的性能和可扩展性，这篇论文通过动态生成原则和批判性反馈，使得奖励模型在推理时能够灵活扩展计算资源。具体的来说，SPCT采用生成式奖励建模（GRM）和在线强化学习，训练模型自适应地生成评判标准和具体评分，从而提高奖励的准确性和泛化能力。

实验表明，经过SPCT训练的DeepSeeK-GRM模型，即使参数量较小（270亿参数），通过并行采样32次并结合奖励模型引导的投票机制，其性能可以显著超过未经SPCT训练的更大规模模型（如671亿参数模型）。

# 不同的奖励模型方法
1. 奖励生成范式
   - 标量
   - 半标量
   - 生成式
2. 评分模式
   - 点式
   - 成对

# SPCT
注意，这里的Reward model是Actor model了，要将Reward model训练的好

SPCT是专门为点式生成式奖励模型而设计的，生成式能提高推理时的可扩展性，通过SPCT，GRM能够自适应地生成高质量的原则，并基于这些原则有效地指导批判生成，从而改善奖励的质量，并为推理时扩展奠定基础。因为模型自身生成的原则最初效果不加，但是经过筛选的高质量原则可以显著提高奖励质量，所以要用特别的训练方法。

## 拒绝式微调
SPCT的冷启动阶段，使得GRM能生成正确的原则和批判，可以

## 基于规则的在线强化学习
目的是改进模型自适应地提出原则和批判能力，从而在通用领域获得更好的奖励结果，目标是鼓励GRM区分最佳回复，奖励信号可以从任何偏好数据集和LLM恢复中获取

精心构建训练数据：
- 对单响应，配对响应和多响应三种格式进行采样（N_RFT=3）
- 采用双重拒绝策略：排除预测奖励与真实奖励不符的轨迹，以及所有N_RFT次评判都正确的“过于简单”样本
- 引入提示采样技术，将最大argmax{r_i}直接加入提示（给差生递答案），引导模型对齐真实标签

## 评判

奖励机制：
Max Voting 和 MetaRM
Max Voting:
从一堆答案中找到最好那个，并且给他的评价最高的，就给个大大的赞+1，否则-1

MetaRM:
MetaRM先选择正确的原则，也需要数据训练，先选择一波比较好的生成的原则和批评好不好，这写比较好的原则和批评再去打分投票，数据集来自RFT阶段

# 思考
这个方法的要点是什么？我们做的时候应该注意哪里？


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Tuple
import numpy as np

"""
伪代码实现论文核心组件：
1. 逐点生成式奖励模型 (Pointwise GRM)
2. 自我原则化批判调优 (SPCT) 训练流程
3. 推理时扩展 (Inference-Time Scaling)
"""


class PointwiseGRM(nn.Module):
    """逐点生成式奖励模型 (基于预训练语言模型)"""

    def __init__(self, model_name: str = "gemma-2b"):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # 分数提取层 (从生成文本中解析分数)
        self.score_extractor = nn.Linear(self.model.config.hidden_size, 1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        """前向传播 (生成原则+批判+分数)"""
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # 获取最后一层隐藏状态
        # [batch, seq_len, hidden_size]
        last_hidden_state = outputs.hidden_states[-1]

        # 提取分数 (示例: 取[CLS]位置的logits)
        scores = self.score_extractor(last_hidden_state[:, 0, :])  # [batch, 1]

        return {
            "logits": outputs.logits,
            "scores": torch.sigmoid(scores) * 10  # 映射到1-10分
        }

    def generate_reward(self, query: str, responses: List[str],
                        max_length: int = 512) -> Dict:
        """
        生成奖励 (原则+批判+分数)
        输入:
            query: 用户查询
            responses: 候选回答列表
        输出:
            {
                "principles": List[str],  # 生成的原则
                "critiques": List[str],   # 生成的批判
                "scores": List[float]     # 提取的分数 (1-10)
            }
        """
        # 构造输入文本 (根据论文附录G的模板)
        input_text = self._build_input_text(query, responses)
        inputs = self.tokenizer(input_text, return_tensors="pt", truncation=True,
                                max_length=max_length, padding="max_length")

        # 生成原则和批判文本
        · = self.model.generate(
            input_ids=inputs["input_ids"].to(self.device),
            attention_mask=inputs["attention_mask"].to(self.device),
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True
        )

        # 解码生成文本
        generated_text = self.tokenizer.decode(
            generated[0], skip_special_tokens=True)

        # 从生成文本中提取原则、批判和分数 (简化实现)
        principles, critiques, scores = self._extract_from_generation(
            generated_text, len(responses))

        return {
            "principles": principles,
            "critiques": critiques,
            "scores": scores
        }

    def _build_input_text(self, query: str, responses: List[str]) -> str:
        """构造模型输入文本 (根据论文附录G模板)"""
        prompt_template = """你是一个熟练的评分专家。请根据给定的评判标准评估以下回答。
给定对话上下文(最后一轮是用户查询)和助手的多个回答，你需要参考[通用评估标准]来评分。
基于通用评估标准，说明针对该查询的其他特定标准、各标准权重，然后给出综合评分。

#### 评估标准 ####
1. 指令遵循 (权重: 30%)
2. 有用性 (权重: 25%)
3. 细节程度 (权重: 20%) 
4. 相关性 (权重: 25%)

#### 对话上下文 ####
{query}

#### 待评分回答 ####
{responses_text}
"""
        responses_text = "\n".join(
            [f"[回答 {i+1}]\n{r}" for i, r in enumerate(responses)])
        return prompt_template.format(query=query, responses_text=responses_text)

    def _extract_from_generation(self, text: str, num_responses: int) -> Tuple:
        """从生成文本中提取原则、批判和分数 (简化实现)"""
        # 实际实现需要更复杂的文本解析逻辑
        principles = ["生成原则1", "生成原则2"]  # 示例
        critiques = [f"批判{i+1}" for i in range(num_responses)]
        scores = np.clip(np.random.rand(num_responses)
                         * 10, 1, 10).tolist()  # 示例随机分数
        return principles, critiques, scores


class SPCTTrainer:
    """自我原则化批判调优(SPCT)训练器"""

    def __init__(self, model: PointwiseGRM, learning_rate: float = 5e-6):
        self.model = model
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=learning_rate)

        # 来自论文C节的超参数
        self.kl_coef = 0.08  # KL散度系数
        self.group_size = 4   # GRPO的组大小

    def rejective_fine_tuning(self, dataset: Dataset, epochs: int = 3):
        """阶段1: 拒绝式微调 (Rejective Fine-Tuning)"""
        dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

        for epoch in range(epochs):
            for batch in dataloader:
                # 论文中的拒绝采样策略 (简化实现)
                valid_indices = self._apply_rejection_strategy(batch)
                if len(valid_indices) == 0:
                    continue

                # 使用有效样本计算损失
                outputs = self.model(
                    input_ids=batch["input_ids"][valid_indices],
                    attention_mask=batch["attention_mask"][valid_indices]
                )

                # 计算损失 (示例: 分数与标签的MSE + 生成文本的交叉熵)
                score_loss = nn.MSELoss()(
                    outputs["scores"], batch["scores"][valid_indices])
                gen_loss = nn.CrossEntropyLoss()(
                    outputs["logits"][:, :-1, :],
                    batch["input_ids"][valid_indices, 1:]
                )
                total_loss = score_loss + gen_loss

                # 反向传播
                self.optimizer.zero_grad()
                total_loss.backward()
                self.optimizer.step()

    def _apply_rejection_strategy(self, batch: Dict) -> List[int]:
        """应用论文中的拒绝策略 (Eq 4)"""
        # 简化实现: 随机选择部分样本
        batch_size = batch["input_ids"].shape[0]
        return np.random.choice(batch_size, size=batch_size//2, replace=False).tolist()

    def rule_based_rl(self, dataset: Dataset, steps: int = 1000):
        """阶段2: 基于规则的强化学习"""
        dataloader = DataLoader(
            dataset, batch_size=self.group_size, shuffle=True)

        for step in range(steps):
            batch = next(iter(dataloader))

            # 生成原则和批判
            with torch.no_grad():
                outputs = self.model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"]
                )
                generated = self.model.generate(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    max_new_tokens=256
                )

            # 计算规则奖励 (Eq 5)
            rewards = self._calculate_rule_rewards(
                generated_scores=outputs["scores"],
                ground_truth=batch["scores"]
            )

            # GRPO 策略优化 (简化实现)
            # 实际实现需要更完整的PPO/GRPO流程
            loss = self._grpo_loss(
                old_logits=outputs["logits"],
                new_logits=self.model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"]
                )["logits"],
                rewards=rewards
            )

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

    def _calculate_rule_rewards(self, generated_scores: torch.Tensor,
                                ground_truth: torch.Tensor) -> torch.Tensor:
        """计算基于规则的奖励 (Eq 5)"""
        # 简化实现: 如果预测分数与真实标签一致则奖励+1，否则-1
        pred_best = torch.argmax(generated_scores, dim=1)
        true_best = torch.argmax(ground_truth, dim=1)
        rewards = (pred_best == true_best).float() * 2 - 1  # 映射到[-1, 1]
        return rewards

    def _grpo_loss(self, old_logits: torch.Tensor, new_logits: torch.Tensor,
                   rewards: torch.Tensor) -> torch.Tensor:
        """GRPO损失函数 (简化实现)"""
        # 计算策略比率
        old_probs = torch.softmax(old_logits, dim=-1)
        new_probs = torch.softmax(new_logits, dim=-1)
        ratio = (new_probs / old_probs).mean(dim=-1)

        # 策略梯度损失
        pg_loss = -torch.mean(ratio * rewards)

        # KL散度惩罚
        kl_div = torch.nn.functional.kl_div(
            torch.log(new_probs + 1e-10),
            old_probs,
            reduction="batchmean"
        )

        return pg_loss + self.kl_coef * kl_div


class InferenceTimeScaling:
    """推理时扩展实现"""

    def __init__(self, grm: PointwiseGRM, meta_rm: PointwiseGRM = None):
        self.grm = grm
        self.meta_rm = meta_rm

    def vote_rewards(self, query: str, responses: List[str],
                     num_samples: int = 8) -> Dict:
        """
        投票聚合奖励 (Eq 6)
        输入:
            query: 用户查询
            responses: 候选回答列表
            num_samples: 采样次数
        输出:
            {
                "principles": List[List[str]],  # 每次采样的原则
                "critiques": List[List[str]],   # 每次采样的批判
                "scores": np.ndarray            # 聚合后的分数 [num_responses]
            }
        """
        all_principles = []
        all_critiques = []
        all_scores = []

        # 并行采样 (简化实现: 实际需要更高效的并行化)
        for _ in range(num_samples):
            result = self.grm.generate_reward(query, responses)
            all_principles.append(result["principles"])
            all_critiques.append(result["critiques"])
            all_scores.append(result["scores"])

        # 转换为numpy数组
        scores_array = np.array(all_scores)  # [num_samples, num_responses]

        # 投票聚合 (求和)
        voted_scores = np.sum(scores_array, axis=0)

        return {
            "principles": all_principles,
            "critiques": all_critiques,
            "scores": voted_scores
        }

    def meta_rm_guided_vote(self, query: str, responses: List[str],
                            num_samples: int = 32, k_meta: int = 16) -> Dict:
        """
        元奖励模型引导投票 (论文4.2节)
        输入:
            query: 用户查询
            responses: 候选回答列表
            num_samples: 总采样次数
            k_meta: 保留的高质量采样数
        输出:
            同vote_rewards
        """
        if self.meta_rm is None:
            raise ValueError("Meta Reward Model not provided")

        all_results = []
        meta_scores = []

        # 生成所有采样
        for _ in range(num_samples):
            result = self.grm.generate_reward(query, responses)

            # 使用Meta RM评估这次采样的质量
            meta_input = self._build_meta_input(query, responses, result)
            meta_score = self.meta_rm.generate_reward(meta_input, [""])[
                "scores"][0]

            all_results.append(result)
            meta_scores.append(meta_score)

        # 选择top-k_meta个高质量采样
        top_indices = np.argsort(meta_scores)[-k_meta:]
        top_results = [all_results[i] for i in top_indices]

        # 聚合分数
        scores_array = np.array([r["scores"] for r in top_results])
        voted_scores = np.sum(scores_array, axis=0)

        return {
            "principles": [r["principles"] for r in top_results],
            "critiques": [r["critiques"] for r in top_results],
            "scores": voted_scores
        }

    def _build_meta_input(self, query: str, responses: List[str],
                          grm_result: Dict) -> str:
        """构造Meta RM的输入文本"""
        principles_text = "\n".join(grm_result["principles"])
        critiques_text = "\n".join([
            f"回答{i+1}批判: {c}"
            for i, c in enumerate(grm_result["critiques"])
        ])

        return f"""评估以下奖励模型生成的原则和批判的质量:
        
原始查询: {query}

生成的原则:
{principles_text}

生成的批判:
{critiques_text}

请评估这些原则和批判是否合理、一致且有助于区分回答质量。给出1-10分的评分。
"""


# 示例使用流程
if __name__ == "__main__":
    # 1. 初始化GRM
    grm = PointwiseGRM("gemma-2b")

    # 2. SPCT训练 (简化示例)
    # 假设我们有一个实现了__getitem__的Dataset类
    # train_dataset = RewardDataset(…)
    # trainer = SPCTTrainer(grm)
    # trainer.rejective_fine_tuning(train_dataset)
    # trainer.rule_based_rl(train_dataset)

    # 3. 推理时扩展
    query = "请解释量子计算的基本原理"
    responses = [
        "量子计算利用量子比特的叠加态和纠缠态进行计算…",
        "量子计算机比传统计算机快很多…",
        "量子计算是一种使用量子力学原理进行计算的新型计算模式…"
    ]

    # 基础推理
    base_result = grm.generate_reward(query, responses)
    print("基础推理结果:", base_result)

    # 投票扩展
    scaling = InferenceTimeScaling(grm)
    voted_result = scaling.vote_rewards(query, responses, num_samples=8)
    print("投票扩展结果:", voted_result["scores"])

    # 元模型引导投票 (假设已训练meta_rm)
    # meta_rm = PointwiseGRM("meta-model")
    # meta_result = scaling.meta_rm_guided_vote(query, responses)
    # print("元模型引导结果:", meta_result["scores"])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import random
import re

class GenerativeRewardModel:
    def __init__(self, model_name="google/gemma-3-1b-it"):
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        print(f"Initialized GRM with {model_name} on {self.device}")

    def _generate_principles(self, query, responses):
        prompt = (
            f"Query: '{query}'\n"
            f"1. {responses[0]}\n"
            f"2. {responses[1]}\n"
            f"List 2 principles to evaluate these (e.g., accuracy, clarity), comma-separated:\n"
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.5,  # Lower temperature for less creativity
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )
        principles = self.tokenizer.decode(outputs[0], skip_special_tokens=True).replace(prompt, "").strip()
        return principles if principles else "accuracy, clarity"

    def generate_critique_and_scores(self, query, responses, generate_principles=True):
        principles = self._generate_principles(query, responses) if generate_principles else "accuracy, clarity"

        prompt = (
            f"Query: '{query}'\n"
            f"1: {responses[0]}\n"
            f"2: {responses[1]}\n"
            f"Principles: {principles}\n"
            f"Score each response (0-10) based on the principles.\n"
            f"Output ONLY:\n"
            f"Analysis: [short analysis]\n"
            f"Scores: \\boxed[num1, num2]\n"
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.5,  # Less randomness
            top_p=0.9,       # Focus on likely tokens
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )
        generation = self.tokenizer.decode(outputs[0], skip_special_tokens=True).replace(prompt, "").strip()
        print(f"Generated evaluation:\n{generation}\n")
        return generation, principles

    def extract_scores_from_generation(self, generation_text):
        try:
            # Look for \boxed[...] first
            match = re.search(r'\\boxed\[([0-9,\s]+)\]', generation_text)
            if match:
                scores = [int(s.strip()) for s in match.group(1).split(',')]
                return scores if len(scores) == 2 else None

            # Fallback: extract any two numbers from 0-10
            numbers = re.findall(r'\b([0-9]|10)\b', generation_text)
            if numbers and len(numbers) >= 2:
                return [int(n) for n in numbers[:2]]  # Take first two numbers
            print("No valid scores found")
            return None
        except Exception as e:
            print(f"Error parsing scores: {str(e)}")
            return None

class SPCTTrainer:
    def __init__(self, grm_model, ref_model=None):
        self.grm_model = grm_model
        self.ref_model = ref_model if ref_model else grm_model

    def _check_correctness(self, predicted_scores, ground_truth_scores):
        if not predicted_scores or not ground_truth_scores or len(predicted_scores) != len(ground_truth_scores):
            return False
        n = len(predicted_scores)
        if n >= 2:
            gt_best_idx = ground_truth_scores.index(max(ground_truth_scores))
            pred_best_idx = predicted_scores.index(max(predicted_scores))
            is_best_correct = (pred_best_idx == gt_best_idx)
            all_others_lower = all(predicted_scores[j] < predicted_scores[pred_best_idx]
                                 for j in range(n) if j != pred_best_idx)
            return is_best_correct and all_others_lower
        return False

    def rejective_fine_tuning(self, dataset, n_samples_per_query=3):
        print("\n--- Starting Rejective Fine-Tuning ---")
        accepted_trajectories = []
        for query, responses, ground_truth_scores in dataset:
            correct_samples = 0
            sampled_trajectories_for_query = []
            for _ in range(n_samples_per_query):
                generation, principles = self.grm_model.generate_critique_and_scores(query, responses)
                pred_scores = self.grm_model.extract_scores_from_generation(generation)
                is_correct = self._check_correctness(pred_scores, ground_truth_scores)
                trajectory = {
                    "query": query,
                    "responses": responses,
                    "principles": principles,
                    "generation": generation,
                    "scores": pred_scores,
                    "is_correct": is_correct
                }
                sampled_trajectories_for_query.append(trajectory)
                if is_correct:
                    correct_samples += 1

            if 0 < correct_samples < n_samples_per_query:
                for traj in sampled_trajectories_for_query:
                    if traj["is_correct"]:
                        accepted_trajectories.append(traj)

        print(f"Collected {len(accepted_trajectories)} trajectories for RFT.")
        return self.grm_model

# Example usage
if __name__ == "__main__":
    grm = GenerativeRewardModel()
    rft_dataset = [("What is AI?",
                    ["AI is artificial intelligence.", "AI is a fruit."],
                    [10, 1])]

    trainer = SPCTTrainer(grm)
    grm_after_rft = trainer.rejective_fine_tuning(rft_dataset)